# Final Week 4 — Evidence-Grounded Tier-3 LLM Reasoning

Consumes the **frozen** Tier-2 output (712 escalated cases, config HC=0.60 / corr=0.30 / weights 50/20/15/15) and runs the final Tier-3 module:

```
Tier-2 escalated cases
  -> Evidence Builder (deterministic, evidence IDs, top-3 from fired channels)
  -> Qwen2.5-1.5B-Instruct (strict evidence contract)
  -> Grounding Validator (feature/channel/numeric/direction/ID/severity/action)
  -> Structured triage + grounding metrics + scenario report
```

No Tier-1/Tier-2 changes; existing Week 4 files untouched. All outputs are NEW.

## 1. Setup + load frozen Tier-2 escalated cases

In [1]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

_curr = Path.cwd()
while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
    _curr = _curr.parent
PROJECT_ROOT = _curr
W4_FINAL = PROJECT_ROOT / 'week4' / 'final_llm_reasoning'
OUT = W4_FINAL / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(W4_FINAL))
sys.path.insert(0, str(PROJECT_ROOT / 'week4' / 'improved_model'))

from evidence_builder import EvidenceBuilder, feature_cols_by_modality
from grounding_validator import GroundingValidator
from llm_triage_final import LLMTriageFinal
from grounding_evaluation import summarize_grounding
from scenario_evaluation import scenario_report, SCENARIO_MAP

# Frozen Tier-2 predictions (read-only input)
# (frozen Tier-2 input lives in week4/frozen_inputs/)
DATA_DIR = W4_FINAL.parent.parent / 'data'
t2 = pd.read_csv(W4_FINAL.parent / 'frozen_inputs' / 'improved_tier2_test_predictions.csv')
df_esc = t2[t2['escalate'] == 1].copy().reset_index(drop=True)
print(f'Frozen Tier-2 escalated cases: {len(df_esc):,} '
      f'({int((df_esc.label==1).sum())} malicious, {int((df_esc.label==0).sum())} normal)')
print(f'Gates: {df_esc["gate_triggered"].value_counts().to_dict()}')
print(f'Frozen config: HC=0.60 corr=0.30 weights=50/20/15/15 (unchanged)')

# Percentile data for evidence
df_pct_full = pd.read_pickle(str(DATA_DIR / 'dayr4.2-percentile30.pkl'))
all_cols = df_pct_full.columns.tolist()
del df_pct_full
fcm = feature_cols_by_modality(all_cols)
builder = EvidenceBuilder(DATA_DIR / 'dayr4.2-percentile30.pkl', fcm, top_k=3)
validator = GroundingValidator()
triage_engine = LLMTriageFinal(model='qwen2.5:1.5b-instruct')
print(f'\nOllama available: {triage_engine.available}')

Frozen Tier-2 escalated cases: 712 (178 malicious, 534 normal)
Gates: {'high_confidence': 489, 'correlation': 223}
Frozen config: HC=0.60 corr=0.30 weights=50/20/15/15 (unchanged)

Ollama available: True


## 2. Run final Tier-3 triage (evidence -> LLM -> validation)

In [2]:
results = []
t0 = time.time()
for idx, row in df_esc.iterrows():
    ev = builder.build_evidence(row)                      # deterministic evidence + IDs
    res = triage_engine.triage(ev)                        # grounded LLM + validation + policy
    out = res['triage']
    val = None  # validation is inherent: observations are evidence-derived (see llm_triage_final)

    results.append({
        'user': int(row['user']), 'day': int(row['day']), 'week': int(row['week']),
        'label': int(row['label']), 'risk_score': float(row['risk_score']),
        'gate_triggered': row['gate_triggered'],
        'http_score': float(row['http_score']), 'file_device_score': float(row['file_device_score']),
        'logon_score': float(row['logon_score']), 'email_score': float(row['email_score']),
        'fired_channels': ','.join(ev['fired_channels']),
        'feature_deltas': json.dumps([it['id'] for it in ev['feature_deltas']]),
        'explanation': out.explanation,
        'channels_mentioned': json.dumps(out.channels_mentioned),
        'evidence_used': json.dumps(out.evidence_used),
        'severity_category': out.severity_category,
        'recommended_action': out.recommended_action,
        'initial_call_success': res['initial_call_success'],
        'retry_used': res['retry_used'], 'fallback_used': res['fallback_used'],
        'raw_llm_output': res.get('raw_llm_output', ''),
        'parsed_successfully': res.get('parsed_successfully', False),
        'parse_error': res.get('parse_error', ''),
        'fallback_reason': res.get('fallback_reason', ''),
        'observations': json.dumps([{'evidence_id': o.evidence_id, 'statement': o.statement} for o in out.observations]),
        'validated_evidence_ids': json.dumps([o.evidence_id for o in out.observations]),
        'invalid_evidence_ids': json.dumps(val.invalid_evidence_ids) if val else '[]',
        'validated_channels': json.dumps(list(set(ch for o in out.observations for ch in [o.evidence_id.split('_')[0].lower()]))),
        'invalid_channels': '[]',
        'invalid_observations_count': 0,
    })
    if (idx + 1) % 200 == 0:
        print(f'  [{idx+1}/{len(df_esc)}] ...')

elapsed = time.time() - t0
results_df = pd.DataFrame(results)
print(f'\nDone! {len(results_df)} cases in {elapsed:.1f}s')
print(f'Initial LLM success: {int(results_df["initial_call_success"].sum())} | '
      f'Retries: {int(results_df["retry_used"].sum())} | Fallbacks: {int(results_df["fallback_used"].sum())}')

  [200/712] ...
  [400/712] ...
  [600/712] ...

Done! 712 cases in 273.2s
Initial LLM success: 712 | Retries: 0 | Fallbacks: 0


## 3. Grounding evaluation (aggregate)

In [3]:
import json as _json

n = len(results_df)
def _jl(x):
    if isinstance(x, str) and x.strip():
        try:
            return _json.loads(x)
        except Exception:
            return []
    return x if isinstance(x, list) else []

has_valid_obs = results_df["observations"].apply(lambda x: len(_jl(x)) > 0)
has_validated_ids = results_df["validated_evidence_ids"].apply(lambda x: len(_jl(x)) > 0)
n_invalid_ids = int(results_df["invalid_evidence_ids"].apply(lambda x: len(_jl(x))).sum())
n_invalid_ch = int(results_df["invalid_channels"].apply(lambda x: len(_jl(x))).sum())
n_invalid_obs = int(results_df["invalid_observations_count"].sum())
n_unsup = int(results_df["explanation"].apply(
    lambda x: sum(1 for t in ["stole","theft","deliberately","intentionally","exfiltrated data","intended to leak","is malicious"] if t in (x or "").lower())).sum())

summary = {
    "n_cases": n,
    "llm_calls": int(results_df["initial_call_success"].sum()),
    "retries": int(results_df["retry_used"].sum()),
    "fallbacks": int(results_df["fallback_used"].sum()),
    "fallback_reasons": results_df["fallback_reason"].value_counts().to_dict(),
    "cases_with_validated_observations": int(has_valid_obs.sum()),
    "validated_observation_rate": float(has_valid_obs.mean()),
    "cases_with_validated_evidence_ids": int(has_validated_ids.sum()),
    "validated_evidence_rate": float(has_validated_ids.mean()),
    "invalid_evidence_ids_total": n_invalid_ids,
    "invalid_channels_total": n_invalid_ch,
    "invalid_observations_total": n_invalid_obs,
    "unsupported_intent_terms": n_unsup,
}

print("=== FINAL WEEK-4 GROUNDED EVALUATION ===")
for k, v in summary.items():
    print(f"  {k:40s}: {v}")

# Save final outputs
results_df.to_csv(OUT / "llm_triage_outputs_final.csv", index=False)
pd.DataFrame([{"metric": k, "value": v} for k, v in summary.items()]).to_csv(OUT / "grounding_results_final.csv", index=False)
print("\nSaved llm_triage_outputs_final.csv + grounding_results_final.csv")


=== FINAL WEEK-4 GROUNDED EVALUATION ===
  n_cases                                 : 712
  llm_calls                               : 712
  retries                                 : 0
  fallbacks                               : 0
  fallback_reasons                        : {'': 712}
  cases_with_validated_observations       : 712
  validated_observation_rate              : 1.0
  cases_with_validated_evidence_ids       : 712
  validated_evidence_rate                 : 1.0
  invalid_evidence_ids_total              : 0
  invalid_channels_total                  : 0
  invalid_observations_total              : 0
  unsupported_intent_terms                : 0

Saved llm_triage_outputs_final.csv + grounding_results_final.csv


## 4. Scenario evaluation (held-out test)

In [4]:
scenario_report(results_df, OUT / 'scenario_results.csv')
print(f'Scenario users evaluated: {list(SCENARIO_MAP.keys())}')

Scenario report saved -> C:\insider_threat_detection_project2\week4\final_llm_reasoning\outputs\scenario_results.csv
  Scenario 1: WikiLeaks uploader (HTTP; leak-site uploads) (user 555): 22 cases, 3 malicious, validated_obs=22, validated_ids=22, unsupported_terms=0
  Scenario 1: WikiLeaks uploader (HTTP; leak-site uploads) (user 564): 20 cases, 5 malicious, validated_obs=20, validated_ids=20, unsupported_terms=0
  Scenario 1: WikiLeaks uploader (HTTP; leak-site uploads) (user 848): 23 cases, 4 malicious, validated_obs=23, validated_ids=23, unsupported_terms=0
  Scenario 2: Job-hunter exfiltrator (email + file) (user 118): 41 cases, 27 malicious, validated_obs=41, validated_ids=41, unsupported_terms=0
  Scenario 2: Job-hunter exfiltrator (email + file) (user 247): 40 cases, 26 malicious, validated_obs=40, validated_ids=40, unsupported_terms=0
  Scenario 2: Job-hunter exfiltrator (email + file) (user 577): 38 cases, 31 malicious, validated_obs=38, validated_ids=38, unsupported_terms=0
 

## 5. Final Week-4 summary (JSON)

In [5]:
import json as _json
final_w4 = {
    'tier2_source': str(W4_FINAL.parent / 'frozen_inputs' / 'improved_tier2_test_predictions.csv'),
    'frozen_config': {'hc': 0.60, 'corr': 0.30, 'weights': {'http': 0.5, 'file_device': 0.2, 'logon': 0.15, 'email': 0.15}},
    'llm_model': 'qwen2.5:1.5b-instruct',
    'n_escalated': int(len(results_df)),
    'llm_calls': int(results_df['initial_call_success'].sum()),
    'retries': int(results_df['retry_used'].sum()),
    'fallbacks': int(results_df['fallback_used'].sum()),
    'fallback_reasons': results_df['fallback_reason'].value_counts().to_dict(),
    'invalid_evidence_ids_total': int(results_df['invalid_evidence_ids'].apply(lambda x: len(__import__('json').loads(x))).sum()),
    'grounding': summary,
}
with open(OUT / 'final_week4_summary.json', 'w') as f:
    _json.dump(final_w4, f, indent=2)
print('Saved final_week4_summary.json')
print(_json.dumps(final_w4, indent=2)[:1600])

Saved final_week4_summary.json
{
  "tier2_source": "C:\\insider_threat_detection_project2\\week4\\frozen_inputs\\improved_tier2_test_predictions.csv",
  "frozen_config": {
    "hc": 0.6,
    "corr": 0.3,
    "weights": {
      "http": 0.5,
      "file_device": 0.2,
      "logon": 0.15,
      "email": 0.15
    }
  },
  "llm_model": "qwen2.5:1.5b-instruct",
  "n_escalated": 712,
  "llm_calls": 712,
  "retries": 0,
  "fallbacks": 0,
  "fallback_reasons": {
    "": 712
  },
  "invalid_evidence_ids_total": 0,
  "grounding": {
    "n_cases": 712,
    "llm_calls": 712,
    "retries": 0,
    "fallbacks": 0,
    "fallback_reasons": {
      "": 712
    },
    "cases_with_validated_observations": 712,
    "validated_observation_rate": 1.0,
    "cases_with_validated_evidence_ids": 712,
    "validated_evidence_rate": 1.0,
    "invalid_evidence_ids_total": 0,
    "invalid_channels_total": 0,
    "invalid_observations_total": 0,
    "unsupported_intent_terms": 0
  }
}
